# whisper-repro on Colab — large-v2 on the constructed corpus

**Runtime → Change runtime type → T4 GPU** before running anything.

Scope tonight is deliberately small: `large-v2` on the 30 utterances of your
own recordings, to answer *does capacity recover Nepali?*. That is one row in
the results table and one paragraph in the discussion. The paper's English
sets are not re-run — you already have published figures for those.


## 1. GPU


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print(torch.__version__, torch.cuda.is_available())


## 2. Mount Drive

Everything the pipeline writes lands here, so a dropped session costs nothing —
`transcribe` skips any hypothesis already on disk. Re-run the cells and it resumes.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Get the code

**Option A (no auth, fastest):** on your laptop run
`git archive -o /tmp/repro.zip HEAD` inside the repo, then drag `repro.zip`
into Colab's Files panel and run the cell as-is.

**Option B:** clone from GitHub. Public repo — just the clone line. Private —
uncomment the token block; `getpass` keeps it out of the saved notebook.


In [ ]:
# --- Option A: uploaded zip ---
!mkdir -p /content/whisper-repro && unzip -oq /content/repro.zip -d /content/whisper-repro

# --- Option B: clone ---
# from getpass import getpass
# import subprocess
# USER = '<your-github-username>'
# tok = getpass('GitHub token (blank if public): ')
# url = f'https://{tok}@github.com/{USER}/whisper-repro.git' if tok else f'https://github.com/{USER}/whisper-repro.git'
# subprocess.run(['git','clone','-q',url,'/content/whisper-repro'], check=True)
# subprocess.run(['git','-C','/content/whisper-repro','remote','set-url','origin',
#                 f'https://github.com/{USER}/whisper-repro.git'], check=True)  # drop the token

%cd /content/whisper-repro
!ls


## 4. Install

`openai-whisper` is pinned to the version behind every result you already have —
a newer decoder would make tonight's numbers incomparable with your `base` runs.

torch and ffmpeg are already on Colab; pip keeps the existing CUDA build rather
than pulling a CPU wheel.


In [ ]:
!pip install -q openai-whisper==20250625
!pip install -q -e .
import whisper, torch
print('torch', torch.__version__, '|', torch.cuda.get_device_name(0))


## 5. Data root on Drive


In [ ]:
%env WHISPER_REPRO_DATA=/content/drive/MyDrive/whisper-repro-data
!mkdir -p "$WHISPER_REPRO_DATA"
!whisper-repro datasets


## 6. Your recordings

Upload your source folder (the one holding `transcripts.csv` and `audio/`) to
Drive first — drag it into My Drive in a browser tab. Then set the path below.

This is the only data that has to come from you; everything else re-downloads.


In [ ]:
SOURCE_DIR = '/content/drive/MyDrive/nepali-source'   # <-- edit

import pathlib
p = pathlib.Path(SOURCE_DIR)
assert (p/'transcripts.csv').exists(), f'transcripts.csv not found under {p}'
n = len(list((p/'audio').glob('*'))) if (p/'audio').is_dir() else len(list(p.glob('*.wav')))
print(f'OK — {n} audio files under {p}')


## 7. Run large-v2

`run` = prepare + transcribe + score. Thirty utterances, a few minutes.


In [ ]:
!whisper-repro run --dataset nepali_custom      --model large-v2 --source-dir "$SOURCE_DIR"
!whisper-repro run --dataset nepali_accented_en --model large-v2 --source-dir "$SOURCE_DIR"


## 8. Compare against base

The number that matters: did large-v2 write any Devanagari at all?


In [ ]:
import csv, json, os, sys
sys.path.insert(0, 'src')
from whisper_repro.scoring.script_id import detect_script
from whisper_repro.paths import results_path, summary_path

for ds in ('nepali_custom', 'nepali_accented_en'):
    for m in ('base', 'large-v2'):
        sp = summary_path(ds, m)
        if not sp.exists():
            continue
        s = json.loads(sp.read_text())
        rows = list(csv.DictReader(results_path(ds, m).open(encoding='utf-8')))
        scripts = {}
        for r in rows:
            k = detect_script(r.get('hypothesis_raw',''))
            scripts[k] = scripts.get(k, 0) + 1
        wer = s.get('wer_norm', s.get('wer_normalised'))
        print(f'{ds:22} {m:9} WER(norm)={wer}  scripts={scripts}')


## 9. Bring it home

Everything is already on Drive under `whisper-repro-data/`. Download
`results/` and `hyps/` from there — a few megabytes of text, and the only
output you can't regenerate for free.

Judge runs locally afterwards: it's API calls, no GPU, and it keeps your
Bedrock key off a shared machine.
